# 1주차 실습 — 토큰화와 서브워드

개념 설명은 [`notes/01-nlp-intro-tokenization.md`](../notes/01-nlp-intro-tokenization.md)

In [1]:
# konlpy는 JDK 필요. 없으면 터미널에서 한 번만:
#   conda install -n nlp-study -c conda-forge openjdk -y
%pip install -q tokenizers transformers sentencepiece konlpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from transformers import AutoTokenizer
from konlpy.tag import Okt


def train_bpe(words, vocab_size):
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    tok.train_from_iterator(
        [" ".join(words)],
        trainers.BpeTrainer(vocab_size=vocab_size, show_progress=False),
    )
    return tok

/Users/kimtaeyeong/miniconda3/envs/nlp-study/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
# 실습 1 — 노트의 손 계산 예제 재현
corpus = ["low"] * 5 + ["lower"] * 2 + ["newest"] * 6 + ["widest"] * 3
bpe = train_bpe(corpus, vocab_size=30)

# ID 순서 = 어휘 추가 순서 = 병합 순서
vocab = sorted(bpe.get_vocab().items(), key=lambda kv: kv[1])
print("문자:", [t for t, _ in vocab if len(t) == 1])
print("병합:", [t for t, _ in vocab if len(t) > 1])

문자: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']
병합: ['es', 'est', 'lo', 'low', 'ew', 'new', 'newest', 'dest', 'idest', 'widest', 'er', 'lower']


In [4]:
# lowest / newer / slowest 는 학습에 없던 단어
for word in ["low", "widest", "lowest", "newer", "slowest"]:
    mark = "학습됨" if word in corpus else "처음봄"
    print(f"{word:9s} {mark}  ->  {bpe.encode(word).tokens}")

low       학습됨  ->  ['low']
widest    학습됨  ->  ['widest']
lowest    처음봄  ->  ['low', 'est']
newer     처음봄  ->  ['new', 'er']
slowest   처음봄  ->  ['s', 'low', 'est']


In [5]:
# 실습 2 — 어휘 크기 ↑ → 시퀀스 ↓, 임베딩 파라미터 ↑
# transformers/multilingual 은 코퍼스에 없는 단어
text = """
natural language processing studies how computers understand human language
machine translation converts text from one language into another language
a tokenizer splits raw text into smaller units called tokens
subword tokenization balances vocabulary size and sequence length
byte pair encoding merges the most frequent adjacent pair repeatedly
neural networks learn representations directly from large amounts of data
an encoder reads the source sentence and a decoder generates the target
attention lets the decoder look back at every encoder position
the transformer replaces recurrence with self attention layers
pretraining on large corpora transfers well to downstream tasks
""".split()
sample = "pretraining transformers for multilingual translation"

for size in [80, 150, 300, 1000]:
    pieces = train_bpe(text * 30, size).encode(sample).tokens
    print(f"vocab {size:>4} | {len(pieces):>2}토큰 | {pieces}")

vocab   80 | 25토큰 | ['p', 're', 'tr', 'a', 'in', 'ing', 'trans', 'f', 'or', 'm', 'ers', 'f', 'or', 'm', 'u', 'l', 't', 'i', 'l', 'in', 'gu', 'al', 'trans', 'l', 'ation']
vocab  150 | 22토큰 | ['p', 're', 'tr', 'ain', 'ing', 'transf', 'or', 'm', 'ers', 'f', 'or', 'm', 'u', 'l', 't', 'i', 'l', 'in', 'gu', 'al', 'trans', 'lation']
vocab  300 | 17토큰 | ['pretraining', 'transf', 'or', 'm', 'ers', 'f', 'or', 'm', 'u', 'l', 't', 'i', 'l', 'in', 'gu', 'al', 'translation']
vocab 1000 | 17토큰 | ['pretraining', 'transf', 'or', 'm', 'ers', 'f', 'or', 'm', 'u', 'l', 't', 'i', 'l', 'in', 'gu', 'al', 'translation']


In [6]:
# 실습 3 — 같은 의미, 다른 토큰 수. 컨텍스트 길이 제한은 동일하다
mbert = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

for lang, sent in [
    ("EN", "I really love studying natural language processing."),
    ("KO", "나는 자연어처리 공부하는 것을 정말 좋아한다."),
]:
    pieces = mbert.tokenize(sent)
    print(f"[{lang}] {len(pieces):2d}토큰  {pieces}")

[EN]  8토큰  ['I', 'really', 'love', 'studying', 'natural', 'language', 'processing', '.']
[KO] 16토큰  ['나는', '자', '##연', '##어', '##처', '##리', '공', '##부', '##하는', '것을', '정', '##말', '좋', '##아', '##한다', '.']


In [7]:
# 실습 4 — 통계적 분절 vs 언어학적 분절
# 첫 실행 시 JVM 경고가 뜨지만 동작에는 문제 없다
okt = Okt()
sent = "나는 자연어처리 공부하는 것을 정말 좋아한다."

print("자연어처리")
print("  mBERT :", mbert.tokenize("자연어처리"))
print("  Okt   :", okt.morphs("자연어처리"))
print()
print(f"  mBERT : {len(mbert.tokenize(sent)):2d}개  {mbert.tokenize(sent)}")
print(f"  Okt   : {len(okt.morphs(sent)):2d}개  {okt.morphs(sent)}")

자연어처리
  mBERT : ['자', '##연', '##어', '##처', '##리']
  Okt   : ['자연어', '처리']

  mBERT : 16개  ['나는', '자', '##연', '##어', '##처', '##리', '공', '##부', '##하는', '것을', '정', '##말', '좋', '##아', '##한다', '.']
  Okt   : 11개  ['나', '는', '자연어', '처리', '공부', '하는', '것', '을', '정말', '좋아한다', '.']


In [8]:
# Okt는 조사를 분리하고 품사까지 붙인다. 대신 언어별 분석기가 필요하다
for word, tag in okt.pos(sent):
    print(f"  {word:6s} {tag}")

  나      Noun
  는      Josa
  자연어    Noun
  처리     Noun
  공부     Noun
  하는     Verb
  것      Noun
  을      Josa
  정말     Noun
  좋아한다   Adjective
  .      Punctuation
